# YOLO26 solar filaments: controlled preprocessing, union targets and PQ calibration

The original notebook scored **0.36** and the CLAHE experiment **0.33** on the public board (user-reported). These numbers do not establish the cause: the preprocessing notebook also changed the training targets. **The union of all annotators is retained**, as requested. Each physical image produces one union mask, then one YOLO polygon per connected component. Final evaluation still uses each annotator's original instances separately.

The conservative default is **raw, native 2048 × 2048 input**. Optional mild CLAHE, disk masking, and radial background correction are experiments, not assumed improvements. Each setting gets a fingerprinted dataset/run directory. The same deterministic transform is applied exactly once at training, validation and test inference. No test annotations or external MAGFiLO labels are read.

Run all cells on Kaggle with two T4 GPUs. One image per GPU, at most four CPU workers, no RAM image cache, bounded CPU tasks, and one image per inference call keep memory controlled. This notebook contains its own implementation; no companion Python files are required. It will train, calibrate, report validation, write the submission, then display **10 validation and 10 test prediction previews**. Scores and trained previews are produced when run; none are invented here.


## Quick repair of an existing CSV (no retraining or model inference)

If the previous submission was rejected for overlapping masks, run **only the next cell** after setting `RUN_REPAIR_EXISTING_CSV=True` and `REPAIR_INPUT_CSV` to that CSV. It imports its own libraries and does not need any trained model, dataset paths or other notebook cells. The source CSV is preserved; upload the new `*_nonoverlap.csv` file.

For each image, this applies the original union -> 8-connected components operation, re-encodes each component using the original COCO counts format, and numbers the rows again. The default repair threshold is 1 pixel to preserve the input foreground; set `min_area=20` in the call to reproduce the original notebook's component filter too. A disk-backed spool handles interleaved image rows without storing all masks in RAM. The repair audits every written RLE for overlap and writes a separate JSON report.

After a fresh run of the full notebook, use its normal submission; the final pipeline now enforces this same non-overlap rule. Old scores selected using the invalid instance mode should be recalibrated; no new score is claimed by repairing the CSV.


In [ ]:
# Standalone recovery: this cell does not train a model or run GPU inference.
import csv
import json
import os
import sqlite3
import tempfile
from contextlib import closing
from pathlib import Path
import cv2
import numpy as np
from pycocotools import mask as mask_utils


def repair_submission_overlaps(input_path, output_path=None, min_area=1):
    input_path = Path(input_path)
    output_path = (Path(output_path) if output_path is not None else
                   input_path.with_name(input_path.stem + "_nonoverlap.csv"))
    if input_path.resolve() == output_path.resolve():
        raise ValueError("Choose a separate output file; the rejected CSV is preserved.")
    if min_area < 1:
        raise ValueError("min_area must be at least 1 pixel.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    csv.field_size_limit(16 * 1024 * 1024)

    def encode_counts(binary_mask):
        # Exact encoding operation from the original 0.36 notebook.
        encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
        return encoded["counts"].decode("utf-8")

    report = dict(input_csv=str(input_path), output_csv=str(output_path),
                  input_rows=0, output_rows=0, images_processed=0,
                  images_with_input_overlaps=0, input_overlap_assignments=0,
                  min_area=int(min_area), non_overlapping_masks=False)
    with tempfile.TemporaryDirectory(prefix="filament_nonoverlap_", dir=output_path.parent) as directory:
        temporary_csv = Path(directory) / "submission.csv"
        with closing(sqlite3.connect(str(Path(directory)/"input.sqlite"))) as connection:
            connection.execute("CREATE TABLE masks (filament_id TEXT PRIMARY KEY, stem TEXT, counts TEXT)")
            with input_path.open(newline="", encoding="utf-8") as stream:
                reader = csv.DictReader(stream)
                if reader.fieldnames != ["filament_id", "segmentation_rle"]:
                    raise ValueError("Expected exactly filament_id,segmentation_rle columns.")
                for row in reader:
                    if set(row) != {"filament_id", "segmentation_rle"} or None in row.values():
                        raise ValueError("Malformed submission row.")
                    fid, counts = row["filament_id"], row["segmentation_rle"]
                    if not fid or not counts or "_" not in fid:
                        raise ValueError("Empty or invalid filament ID / RLE counts.")
                    stem, suffix = fid.rsplit("_", 1)
                    if not stem or not suffix:
                        raise ValueError(f"Invalid filament ID: {fid}")
                    try:
                        connection.execute("INSERT INTO masks VALUES (?, ?, ?)", (fid, stem, counts))
                    except sqlite3.IntegrityError as error:
                        raise ValueError(f"Duplicate filament ID: {fid}") from error
                    report["input_rows"] += 1
            connection.execute("CREATE INDEX masks_stem ON masks(stem)")
            connection.commit()
            with temporary_csv.open("w", newline="", encoding="utf-8") as stream:
                writer = csv.writer(stream)  # Same default writer as original notebook.
                writer.writerow(["filament_id", "segmentation_rle"])
                for (stem,) in connection.execute("SELECT DISTINCT stem FROM masks ORDER BY stem"):
                    union = np.zeros((2048, 2048), np.uint8)
                    overlap_assignments = 0
                    for (counts,) in connection.execute("SELECT counts FROM masks WHERE stem=?", (stem,)):
                        rle = {"size": [2048, 2048], "counts": counts.encode("utf-8")}
                        mask = mask_utils.decode(rle).astype(np.uint8)
                        if mask.shape != (2048, 2048) or not mask.any():
                            raise ValueError(f"Empty or invalid mask for {stem}")
                        overlap_assignments += int(np.count_nonzero(union & mask))
                        union |= mask
                    number, labels, stats, _ = cv2.connectedComponentsWithStats(union, connectivity=8)
                    keep = [i for i in range(1, number) if stats[i, cv2.CC_STAT_AREA] >= min_area]
                    keep.sort(key=lambda i: (-int(stats[i, cv2.CC_STAT_AREA]),
                                              int(stats[i, cv2.CC_STAT_TOP]), int(stats[i, cv2.CC_STAT_LEFT])))
                    for counter, label in enumerate(keep, start=1):
                        writer.writerow([f"{stem}_{counter}", encode_counts(labels == label)])
                    report["images_processed"] += 1
                    report["output_rows"] += len(keep)
                    report["images_with_input_overlaps"] += int(overlap_assignments > 0)
                    report["input_overlap_assignments"] += overlap_assignments

        # Independently verify the actual written RLEs before publishing the CSV.
        occupied = {}
        seen = set()
        with temporary_csv.open(newline="", encoding="utf-8") as stream:
            for row in csv.DictReader(stream):
                fid = row["filament_id"]
                if fid in seen:
                    raise ValueError(f"Duplicate output ID: {fid}")
                seen.add(fid)
                stem = fid.rsplit("_", 1)[0]
                rle = {"size": [2048, 2048], "counts": row["segmentation_rle"].encode("utf-8")}
                mask = mask_utils.decode(rle)
                if mask.shape != (2048, 2048) or not mask.any() or encode_counts(mask) != row["segmentation_rle"]:
                    raise ValueError(f"Invalid repaired mask: {fid}")
                if stem in occupied:
                    overlap = mask_utils.merge([occupied[stem], rle], intersect=True)
                    if int(mask_utils.area(overlap)):
                        raise ValueError(f"Repaired masks still overlap: {fid}")
                    occupied[stem] = mask_utils.merge([occupied[stem], rle])
                else:
                    occupied[stem] = rle
        if len(seen) != report["output_rows"]:
            raise ValueError("Repaired CSV row-count mismatch.")
        os.replace(temporary_csv, output_path)
    report["non_overlapping_masks"] = True
    report["output_overlap_pixels"] = 0
    output_path.with_suffix(".audit.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    return output_path, report


RUN_REPAIR_EXISTING_CSV = False
REPAIR_INPUT_CSV = Path("solar_filament_yolo26s_submission.csv")  # Or Path("submission.csv").
if RUN_REPAIR_EXISTING_CSV:
    repaired_csv_path, repair_report = repair_submission_overlaps(REPAIR_INPUT_CSV)
    print(json.dumps(repair_report, indent=2))
    print(f"Upload this corrected CSV: {repaired_csv_path}")


## What is correct in the quoted advice?

| Advice                                                         | Assessment for this notebook                                                                                                                                                                                                 |
| -------------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| Disk masking / background flattening                           | Plausible, but an inaccurate radius can erase limb filaments. Fit from pixels, expand the boundary, and fall back to the original image if uncertain. Postprocessing clipping competes with no clipping on calibration data. |
| CLAHE always helps / histogram matching is required            | Unsupported. Strong local equalization can amplify texture, sunspots and seeing artifacts. Compare raw against a mild, blended transform with identical targets/splits. No universal 800/1024 size is established.           |
| Correct differential rotation / apply SIDE                     | Research-specific, not a requirement for independent JPEG segmentation. Spatial warps would need corresponding label transforms. Neither is introduced.                                                                      |
| Soft targets from annotator agreement                          | Reasonable for a custom semantic loss, but stock YOLO polygon labels do not encode soft probabilities. The requested union is a hard target, not consensus.                                                                  |
| YOLO guarantees one instance / U-Net cannot separate instances | Incorrect as a guarantee. Either approach needs an instance strategy; YOLO can still duplicate, merge or fragment predictions.                                                                                               |
| `retina_masks=True` restores lost detail                       | It decodes masks at the original image size; it cannot recreate missing learned detail. Training `mask_ratio` controls target rasterization, not the prototype architecture.                                                 |
| `mask=32` / “higher overlap_mask=False”                        | `mask` is not a supported loss-gain argument in the pinned release; `overlap_mask` is boolean. Keep supported defaults.                                                                                                      |
| Fixed confidence .25 / NMS .45 / 70% positive crops            | Hypotheses to validate, not competition-proven optima. Full disks preserve whole instances; no crop sampler is added.                                                                                                        |
| Claimed leaderboard recipes and mandatory SAM refinement       | Not established by the cited material. SAM would require its own measured training/refinement experiment.                                                                                                                    |

Sources: [competition overview and Aug 7 PQ update](https://www.kaggle.com/competitions/filament-segmentation-2026/overview), [organizers' self-evaluation](https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook), [MAGFiLO authors](https://www.mlecofi.net/magfilo), [solar detection study](https://arxiv.org/abs/2402.15407), [Ultralytics configuration](https://github.com/ultralytics/ultralytics/blob/v8.4.171/ultralytics/cfg/default.yaml), [native mask decoder](https://github.com/ultralytics/ultralytics/blob/v8.4.171/ultralytics/utils/ops.py).

The submitted CSV specification is unchanged by the metric update. The quoted mean-Dice text predates the organizers' switch to PQ. We reproduce their published self-evaluation rule: **all IoU > 0.5 hits**, unmatched columns/rows as FP/FN, pooled over annotator-image records. This differs from enforcing one-to-one assignment when masks overlap. Local validation is not a leaderboard score.


## 1. Dependencies

Pin Ultralytics to **8.4.171**, the version recorded in the original 0.36 notebook, to avoid silently changing model/loss behavior. Keep Kaggle's existing CUDA-enabled Torch. Resolved package versions and the complete experiment configuration are saved with the run.


In [ ]:
%pip -q install --no-cache-dir ultralytics==8.4.171 pycocotools scikit-learn scipy tqdm threadpoolctl psutil


In [ ]:
from pathlib import Path
import csv, gc, hashlib, json, os, random, re, shutil, time
from collections import deque
from concurrent.futures import ThreadPoolExecutor
from importlib.metadata import version

for _thread_key in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[_thread_key] = "1"  # Also inherited by Ultralytics DDP subprocesses.
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
import ultralytics
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from sklearn.model_selection import GroupShuffleSplit
from threadpoolctl import threadpool_limits
from tqdm.auto import tqdm
from ultralytics import YOLO

assert (
    ultralytics.__version__ == "8.4.171"
), "Restart the kernel after installing the pinned version."
csv.field_size_limit(16 * 1024 * 1024)
cv2.setNumThreads(1)
torch.set_num_threads(1)
CPU_THREAD_LIMIT = threadpool_limits(limits=1)

## 2. Configuration

Start with `raw`. Then run `radial` and `clahe` separately from the same pretrained weights; compare the saved report PQ using the same split. The CLAHE option uses clip 1.2 and only 25% enhancement instead of replacing the full image with clip 2.0 equalization. `radial` estimates an annular quiet-Sun profile with bounded correction; it is an approximation, not an instrument calibration. Failed disk fits leave the image unchanged.

Training retains 2048 inputs and the annotator union. Flips avoid extra interpolation of thin targets; arbitrary rotations and strong brightness jitter are removed from the default. `augmentations=[]` disables Ultralytics' optional Albumentations defaults, which otherwise add blur and CLAHE even to a raw experiment. AdamW, cosine decay and no mosaic/mixup are retained. `mask_ratio=4` and `overlap_mask=True` match the original supported defaults; reducing mask_ratio alone does not increase inference mask resolution.

By default, dates are kept together across **80% train / 10% calibration / 10% report** (approximately, by groups). YOLO early stopping and postprocessing selection see only calibration. The report partition is evaluated after checkpoint/settings selection. `SPLIT_GROUP='filename'` is available for comparison to the original filename split, but can leak observations of the same day. Split changes mean old local validation scores are not directly comparable.


In [ ]:
SEED = 42
DATA_ROOT_OVERRIDE = ""  # Optional explicit competition root.
OUTPUT_ROOT = Path.cwd()
PREPROCESSING_METHOD = "raw"  # raw, disk_raw, radial, clahe, percentile, gamma
PREPROCESSING_CONFIG = {
    "clahe_clip": 1.2,
    "clahe_grid": [16, 16],
    "clahe_strength": 0.25,
    "radial_strength": 0.35,
    "radial_gain_limits": [0.80, 1.25],
    "percentiles": [0.5, 99.5],
    "gamma": 0.9,
    "disk_margin": 1.02,
}
SPLIT_GROUP = "date"
VALIDATION_FRACTION = 0.20
CALIBRATION_SHARE = 0.50
IMAGE_SIZE = 2048
YOLO_MODEL_NAME = "yolo26s-seg.pt"
NUM_EPOCHS = 150
PATIENCE = 25
TARGET_MIN_AREA = 20  # Same union-component threshold as the supplied notebook.
NUM_WORKERS = min(4, os.cpu_count() or 1)
GPU_IDS = list(range(min(2, torch.cuda.device_count())))
TRAIN_DEVICE = GPU_IDS if len(GPU_IDS) > 1 else (GPU_IDS[0] if GPU_IDS else "cpu")
INFERENCE_DEVICE = GPU_IDS[0] if GPU_IDS else "cpu"
BATCH_SIZE = max(1, len(GPU_IDS))
YOLO_WORKERS = min(2, NUM_WORKERS // max(1, len(GPU_IDS))) if GPU_IDS else 0
MAX_DETECTIONS = 100
RAM_RESERVE_GIB = 6.0
DISK_RESERVE_GIB = 6.0
DISK_LIMIT_GIB = 57.6
RUN_TRAINING = True
EXISTING_CHECKPOINT = (
    ""  # Only used if RUN_TRAINING=False; matching run_config.json required.
)
CREATE_SUBMISSION = True
REBUILD_YOLO_DATASET = (
    False  # Fingerprints prevent reusing another experiment's dataset.
)
PREDICTION_PREVIEW_COUNT = 10
PREPROCESS_PREVIEW_COUNT = 3
CONFIDENCE_CANDIDATES = [round(float(v), 2) for v in np.arange(0.05, 0.61, 0.05)]
AREA_CANDIDATES = [0, 20, 80]
POSTPROCESS_MODES = ["union_cc"]  # Original, disjoint connected-component masks only.
CLIP_TO_DISK_CANDIDATES = [False, True]
PREDICT_NMS_IOU = (
    0.70  # Fixed; not described as an optimal or instance-merging threshold.
)
PQ_MATCH_IOU = 0.50
GIB = 1024**3

TRAIN_OPTIONS = dict(
    epochs=NUM_EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=TRAIN_DEVICE,
    workers=YOLO_WORKERS,
    patience=PATIENCE,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    amp=bool(GPU_IDS),
    seed=SEED,
    deterministic=True,
    max_det=MAX_DETECTIONS,
    fliplr=0.5,
    flipud=0.5,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.10,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    close_mosaic=0,
    overlap_mask=True,
    mask_ratio=4,
    augmentations=[],
    plots=True,
    verbose=True,
)
assert PREPROCESSING_METHOD in {
    "raw",
    "disk_raw",
    "radial",
    "clahe",
    "percentile",
    "gamma",
}
assert SPLIT_GROUP in {"date", "filename"}
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if GPU_IDS:
    torch.cuda.manual_seed_all(SEED)
print(
    "Train devices:",
    TRAIN_DEVICE,
    "| batch:",
    BATCH_SIZE,
    "| inference:",
    INFERENCE_DEVICE,
)
if not GPU_IDS and RUN_TRAINING:
    print(
        "CPU detected. Full-resolution 150-epoch training requires a GPU for practical runtime."
    )


In [ ]:
def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=True), encoding="utf-8")
    os.replace(tmp, path)


def fingerprint(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def find_dataset_root():
    expected = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
    roots = (
        [Path(DATA_ROOT_OVERRIDE)]
        if DATA_ROOT_OVERRIDE
        else [Path.cwd(), Path.cwd().parent, Path("/kaggle/input")]
    )
    for parent in roots:
        if not parent.is_dir():
            continue
        for annotation in parent.glob("**/" + expected):
            root = annotation.parent.parent
            if (root / "train/train_images").is_dir() and (
                root / "test/test_images"
            ).is_dir():
                return root.resolve()
    raise FileNotFoundError("Attach the competition dataset or set DATA_ROOT_OVERRIDE.")


def read_gray(path):
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise OSError(f"Unreadable image: {path}")
    if image.shape != (IMAGE_SIZE, IMAGE_SIZE):
        raise ValueError(
            f"Expected native {IMAGE_SIZE} square image, got {image.shape}: {path}"
        )
    return image


def bounded_map(function, items, workers=NUM_WORKERS):
    iterator = iter(items)
    with ThreadPoolExecutor(max_workers=workers) as pool:
        pending = deque()
        for _ in range(workers):
            item = next(iterator, None)
            if item is not None:
                pending.append(pool.submit(function, item))
        while pending:
            yield pending.popleft().result()  # Propagate every worker failure.
            item = next(iterator, None)
            if item is not None:
                pending.append(pool.submit(function, item))


DATA_ROOT = find_dataset_root()
TRAIN_IMAGE_DIR = DATA_ROOT / "train/train_images"
TEST_IMAGE_DIR = DATA_ROOT / "test/test_images"
ANNOTATION_PATH = DATA_ROOT / "train/MAGFiLO_1.0_Annotations_kaggle2026_train.json"
coco = COCO(str(ANNOTATION_PATH))
records = pd.DataFrame(
    [
        dict(
            image_id=i,
            file_name=m["file_name"],
            date=str(m.get("date_captured", ""))[:10],
        )
        for i, m in coco.imgs.items()
    ]
)
if SPLIT_GROUP == "date":
    records["group"] = records["file_name"].map(
        lambda n: re.match(r"^(\d{8})", Path(n).name).group(1)
    )
else:
    records["group"] = records["file_name"]
splitter = GroupShuffleSplit(
    n_splits=1, test_size=VALIDATION_FRACTION, random_state=SEED
)
train_idx, holdout_idx = next(splitter.split(records, groups=records["group"]))
train_records = records.iloc[train_idx].reset_index(drop=True)
holdout = records.iloc[holdout_idx].reset_index(drop=True)
splitter2 = GroupShuffleSplit(
    n_splits=1, train_size=CALIBRATION_SHARE, random_state=SEED + 1
)
cal_idx, report_idx = next(splitter2.split(holdout, groups=holdout["group"]))
calibration_records = holdout.iloc[cal_idx].reset_index(drop=True)
report_records = holdout.iloc[report_idx].reset_index(drop=True)
for left, right in [
    (train_records, calibration_records),
    (train_records, report_records),
    (calibration_records, report_records),
]:
    assert set(left.file_name).isdisjoint(right.file_name)
    assert set(left.group).isdisjoint(right.group)


def physical_rows(frame):
    return (
        frame.groupby("file_name", sort=True)["image_id"]
        .agg(list)
        .reset_index(name="image_ids")
        .to_dict("records")
    )


VERSIONS = {
    p: version(p)
    for p in [
        "ultralytics",
        "torch",
        "torchvision",
        "numpy",
        "opencv-python",
        "pycocotools",
        "scikit-learn",
        "scipy",
    ]
}
RUN_CONFIG = dict(
    pipeline="union-preprocessing-pq-v2",
    seed=SEED,
    image_size=IMAGE_SIZE,
    preprocessing_method=PREPROCESSING_METHOD,
    preprocessing=PREPROCESSING_CONFIG,
    target="all_annotators_union_connected_components",
    target_min_area=TARGET_MIN_AREA,
    split_group=SPLIT_GROUP,
    annotation_sha256=file_sha256(ANNOTATION_PATH),
    train=sorted(train_records.file_name.unique()),
    calibration=sorted(calibration_records.file_name.unique()),
    report=sorted(report_records.file_name.unique()),
    model=YOLO_MODEL_NAME,
    training=TRAIN_OPTIONS,
    versions=VERSIONS,
    selection=dict(
        confidences=CONFIDENCE_CANDIDATES,
        areas=AREA_CANDIDATES,
        modes=POSTPROCESS_MODES,
        disk_clipping=CLIP_TO_DISK_CANDIDATES,
        predict_nms_iou=PREDICT_NMS_IOU,
        pq_match_iou=PQ_MATCH_IOU,
    ),
)
RUN_FINGERPRINT = fingerprint(RUN_CONFIG)
EXPERIMENT_DIR = (
    OUTPUT_ROOT
    / f"preprocessing_{PREPROCESSING_METHOD}_union_v2_{RUN_FINGERPRINT[:10]}"
)
YOLO_DATA_DIR = EXPERIMENT_DIR / "yolo_filament_dataset"
RUNS_DIR = EXPERIMENT_DIR / "yolo_runs"
RUN_NAME = "yolo26s_full_resolution"
DATA_YAML = YOLO_DATA_DIR / "filament.yaml"
PREVIEW_DIR = EXPERIMENT_DIR / "previews"
SUBMISSION_PATH = Path.cwd() / "solar_filament_yolo26s_submission.csv"
save_json(EXPERIMENT_DIR / "run_config.json", RUN_CONFIG)
for label, frame in [
    ("train", train_records),
    ("calibration", calibration_records),
    ("report", report_records),
]:
    print(
        label,
        ":",
        frame.file_name.nunique(),
        "physical images,",
        len(frame),
        "annotator records,",
        frame.group.nunique(),
        "groups",
    )
    frame.to_csv(EXPERIMENT_DIR / f"{label}_split.csv", index=False)
print("Dataset:", DATA_ROOT, "\nOutputs:", EXPERIMENT_DIR)


## 3. Safe image conditioning

Disk detection uses a small image only to fit the geometry; **the model image and labels are never cropped or resized**. Fits must pass center, axis-ratio, radius, and residual checks. All transforms use image pixels only. A 2% radius margin protects limb structures. Radial correction is blended and gain-limited to avoid extreme limb amplification. Raw remains byte-for-byte unchanged.


In [ ]:
def estimate_disk(image):
    h, w = image.shape
    small = cv2.resize(image, (256, 256), interpolation=cv2.INTER_AREA)
    blurred = cv2.GaussianBlur(small, (5, 5), 0)
    _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    if len(contour) < 20 or cv2.contourArea(contour) < 0.15 * small.size:
        return None
    (cx, cy), (axis1, axis2), _ = cv2.fitEllipse(contour)
    radius = (axis1 + axis2) / 4
    points = contour[:, 0, :].astype(np.float32)
    residual = np.percentile(
        np.abs(np.linalg.norm(points - [cx, cy], axis=1) - radius), 90
    )
    good = (
        64 < radius < 132
        and abs(cx - 127.5) < 32
        and abs(cy - 127.5) < 32
        and min(axis1, axis2) / max(axis1, axis2) > 0.90
        and residual / radius < 0.06
        and points.min() > 0
        and points.max() < 255
    )
    if not good:
        return None
    return {
        "cx": float(cx * w / 256),
        "cy": float(cy * h / 256),
        "radius": float(radius * min(h, w) / 256),
        "relative_residual": float(residual / radius),
    }


def disk_radius_grid(shape, disk):
    yy, xx = np.ogrid[: shape[0], : shape[1]]
    return (
        np.sqrt((xx - disk["cx"]) ** 2 + (yy - disk["cy"]) ** 2).astype(np.float32)
        / disk["radius"]
    )


def disk_support(shape, disk, margin=None):
    if disk is None:
        return np.ones(shape, dtype=bool)
    return disk_radius_grid(shape, disk) <= (
        PREPROCESSING_CONFIG["disk_margin"] if margin is None else margin
    )


def preprocess_grayscale(image, method=None, config=None):
    method = PREPROCESSING_METHOD if method is None else method
    cfg = PREPROCESSING_CONFIG if config is None else config
    if image.ndim != 2 or image.dtype != np.uint8:
        raise ValueError("Preprocessing expects uint8 grayscale.")
    if method == "raw":
        return image.copy()
    disk = estimate_disk(image)
    if disk is None:
        return image.copy()
    rr = disk_radius_grid(image.shape, disk)
    support = rr <= cfg["disk_margin"]
    taper = np.clip((1.02 - rr) / 0.12, 0, 1)
    original = image.astype(np.float32)
    if method == "disk_raw":
        result = np.where(support, image, 0)
    elif method == "clahe":
        enhanced = cv2.createCLAHE(
            clipLimit=cfg["clahe_clip"], tileGridSize=tuple(cfg["clahe_grid"])
        ).apply(image)
        alpha = cfg["clahe_strength"] * taper
        result = original * (1 - alpha) + enhanced.astype(np.float32) * alpha
    elif method == "radial":
        # Estimate broad illumination from annular medians, not dark filament pixels.
        small = cv2.resize(image, (512, 512), interpolation=cv2.INTER_AREA)
        scaled = {
            **disk,
            "cx": disk["cx"] * 512 / image.shape[1],
            "cy": disk["cy"] * 512 / image.shape[0],
            "radius": disk["radius"] * 512 / min(image.shape),
        }
        rsmall = disk_radius_grid(small.shape, scaled)
        centers = (np.arange(64) + 0.5) / 64
        profile = np.full(64, np.nan, dtype=np.float32)
        for i in range(64):
            pixels = small[(rsmall >= i / 64) & (rsmall < (i + 1) / 64)]
            if pixels.size >= 16:
                profile[i] = np.median(pixels)
        valid = np.isfinite(profile) & (profile > 5)
        if valid.sum() < 32:
            return image.copy()
        profile = np.interp(centers, centers[valid], profile[valid]).astype(np.float32)
        profile = cv2.GaussianBlur(profile[None, :], (9, 1), 0).ravel()
        target = float(np.median(profile[centers < 0.65]))
        background = np.interp(rr.ravel(), centers, profile).reshape(image.shape)
        gain = np.clip(target / np.maximum(background, 10), *cfg["radial_gain_limits"])
        result = original * (1 + cfg["radial_strength"] * taper * (gain - 1))
    elif method == "percentile":
        low, high = np.percentile(image[rr < 0.95], cfg["percentiles"])
        enhanced = np.clip((original - low) * 255 / max(float(high - low), 1), 0, 255)
        result = original * (1 - taper) + enhanced * taper
    elif method == "gamma":
        enhanced = 255 * (original / 255) ** cfg["gamma"]
        result = original * (1 - taper) + enhanced * taper
    else:
        raise ValueError(f"Unknown method: {method}")
    return np.ascontiguousarray(np.clip(np.rint(result), 0, 255).astype(np.uint8))

## 4. Union targets and dataset preparation

Every annotator of the same training/calibration filename contributes to its union. Components below 20 native pixels are dropped, as before. Nearby instances that touch after union become one training component; this is an explicit consequence of the requested target policy. Evaluation GT is never unioned or area-filtered. Polygon conversion can fill holes and cannot exactly preserve all raster detail; previews show the actual exported target.

Preparation uses bounded threads (also works in Windows notebooks). Missing/corrupt images raise errors. A manifest prevents stale data reuse. Image copies are lossless grayscale PNG; YOLO reads three identical channels. Only train and calibration images are exported; report images are never provided to the trainer.


In [ ]:
def merged_annotation_mask(coco_api, image_ids):
    first = coco_api.imgs[image_ids[0]]
    shape = (first["height"], first["width"])
    merged = np.zeros(shape, np.uint8)
    for image_id in image_ids:
        metadata = coco_api.imgs[image_id]
        if (
            metadata["file_name"] != first["file_name"]
            or (metadata["height"], metadata["width"]) != shape
        ):
            raise ValueError("Union records must describe the same physical image.")
        for ann in coco_api.loadAnns(coco_api.getAnnIds(imgIds=[image_id])):
            merged |= coco_api.annToMask(ann).astype(np.uint8)
    return merged


def mask_to_yolo_segments(mask, min_area=TARGET_MIN_AREA):
    h, w = mask.shape
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    segments = []
    for idx in range(1, count):
        if stats[idx, cv2.CC_STAT_AREA] < min_area:
            continue
        x, y, bw, bh, _ = stats[idx]
        crop = (labels[y : y + bh, x : x + bw] == idx).astype(np.uint8)
        contours, _ = cv2.findContours(crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        if not contours:
            continue
        contour = max(contours, key=cv2.contourArea)
        if len(contour) < 3 or cv2.contourArea(contour) <= 0 or bw < 2 or bh < 2:
            continue
        points = contour[:, 0, :].astype(np.float32) + np.array([x, y], np.float32)
        points /= np.array([w, h], np.float32)
        segments.append(np.clip(points, 0, 1))
    return segments


def resource_preflight():
    available = psutil.virtual_memory().available / GIB
    # Honor container limits when present (Kaggle / Linux).
    try:
        limit = int(Path("/sys/fs/cgroup/memory.max").read_text())
        used = int(Path("/sys/fs/cgroup/memory.current").read_text())
        available = min(available, (limit - used) / GIB)
    except (OSError, ValueError):
        pass
    if available < RAM_RESERVE_GIB + 2:
        raise MemoryError(
            f"Only {available:.1f} GiB available; free memory before proceeding."
        )
    count = len(physical_rows(train_records)) + len(physical_rows(calibration_records))
    copies = count * IMAGE_SIZE**2 * 1.1 / GIB
    cache = count * IMAGE_SIZE**2 * 3 * 1.25 / GIB
    usage = shutil.disk_usage(EXPERIMENT_DIR)
    existing = (
        sum(
            p.stat().st_size
            for p in EXPERIMENT_DIR.rglob("*")
            if p.is_file() and not p.is_symlink()
        )
        / GIB
    )
    budget = min(usage.free / GIB, DISK_LIMIT_GIB - existing)
    # Reserve 2 GiB for checkpoints, prediction caches and previews.
    required = copies + 2 + DISK_RESERVE_GIB
    if budget < required:
        raise OSError(f"Need {required:.1f} GiB disk headroom, have {budget:.1f}.")
    cache_mode = "disk" if budget >= required + cache else False
    print(
        f"Available RAM {available:.1f} GiB; disk budget {budget:.1f} GiB; image cache={cache_mode}"
    )
    return cache_mode


def prepare_record(task):
    row, split_name = task
    image = read_gray(TRAIN_IMAGE_DIR / row["file_name"])
    union = merged_annotation_mask(coco, row["image_ids"])
    if union.shape != image.shape:
        raise ValueError("Annotation/image dimension mismatch.")
    stem = Path(row["file_name"]).stem
    image_path = YOLO_DATA_DIR / "images" / split_name / f"{stem}.png"
    label_path = YOLO_DATA_DIR / "labels" / split_name / f"{stem}.txt"
    processed = preprocess_grayscale(image)
    if not cv2.imwrite(str(image_path), processed, [cv2.IMWRITE_PNG_COMPRESSION, 3]):
        raise OSError(f"Could not save {image_path}")
    segments = mask_to_yolo_segments(union)
    lines = ["0 " + " ".join(f"{v:.7f}" for v in points.ravel()) for points in segments]
    label_path.write_text("\n".join(lines) + ("\n" if lines else ""), encoding="utf-8")
    return dict(
        file_name=row["file_name"],
        split=split_name,
        annotators=len(row["image_ids"]),
        union_pixels=int(union.sum()),
        training_instances=len(segments),
    )


YOLO_CACHE_MODE = resource_preflight()
manifest_path = YOLO_DATA_DIR / "manifest.json"
if REBUILD_YOLO_DATASET and YOLO_DATA_DIR.exists():
    # Deletion is confined to this fingerprinted experiment's generated dataset.
    resolved = YOLO_DATA_DIR.resolve()
    if (
        resolved.parent != EXPERIMENT_DIR.resolve()
        or resolved.name != "yolo_filament_dataset"
    ):
        raise ValueError("Refusing to remove an unexpected dataset path.")
    shutil.rmtree(resolved)
if manifest_path.is_file():
    assert json.loads(manifest_path.read_text())["fingerprint"] == RUN_FINGERPRINT
    print("Reusing completed dataset:", YOLO_DATA_DIR)
else:
    if YOLO_DATA_DIR.exists() and any(YOLO_DATA_DIR.iterdir()):
        raise RuntimeError(
            "Incomplete dataset. Set REBUILD_YOLO_DATASET=True to rebuild this generated directory."
        )
    tasks = []
    for name, frame in [("train", train_records), ("val", calibration_records)]:
        for sub in ["images", "labels"]:
            (YOLO_DATA_DIR / sub / name).mkdir(parents=True, exist_ok=True)
        tasks.extend((row, name) for row in physical_rows(frame))
    stats = list(
        tqdm(bounded_map(prepare_record, tasks), total=len(tasks), desc="Union dataset")
    )
    pd.DataFrame(stats).to_csv(EXPERIMENT_DIR / "target_audit.csv", index=False)
    DATA_YAML.write_text(
        f"path: {json.dumps(YOLO_DATA_DIR.as_posix())}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n",
        encoding="utf-8",
    )
    save_json(manifest_path, {"fingerprint": RUN_FINGERPRINT, "samples": len(tasks)})
for name, frame in [("train", train_records), ("val", calibration_records)]:
    expected = {Path(n).stem for n in frame.file_name.unique()}
    for sub, ext in [("images", ".png"), ("labels", ".txt")]:
        actual = {p.stem for p in (YOLO_DATA_DIR / sub / name).glob("*" + ext)}
        assert actual == expected, f"Incomplete or stale {name} {sub}."
print(DATA_YAML.read_text())

In [ ]:
def pick_evenly(items, count):
    items = list(items)
    return (
        [
            items[int(i)]
            for i in np.linspace(0, len(items) - 1, min(count, len(items)), dtype=int)
        ]
        if items
        else []
    )


def show_gray(ax, image):
    ax.imshow(image, cmap="gray", vmin=0, vmax=255)
    ax.axis("off")


def overlay(ax, image, masks):
    # One RGB display array per panel. A separate RGBA artist per instance
    # would retain gigabytes for a crowded 2048 image.
    rgb = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    for i, mask in enumerate(masks):
        color = np.array(plt.get_cmap("tab20")(i % 20)[:3]) * 255
        rgb[mask] = np.rint(rgb[mask] * 0.55 + color * 0.45).astype(np.uint8)
    ax.imshow(rgb, interpolation="nearest")
    ax.axis("off")


def finish_figure(fig, stage, filename):
    destination = PREVIEW_DIR / stage / f"{Path(filename).stem}.png"
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(destination, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for row in pick_evenly(physical_rows(train_records), PREPROCESS_PREVIEW_COUNT):
    image = read_gray(TRAIN_IMAGE_DIR / row["file_name"])
    union = merged_annotation_mask(coco, row["image_ids"]).astype(bool)
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, method in zip(axes.ravel()[:4], ["raw", "disk_raw", "radial", "clahe"]):
        show_gray(ax, preprocess_grayscale(image, method))
        ax.set_title(method + (" (selected)" if method == PREPROCESSING_METHOD else ""))
    overlay(axes[1, 1], image, [union])
    axes[1, 1].set_title(f'Union of {len(row["image_ids"])} annotator records')
    target = np.zeros(image.shape, np.uint8)
    path = YOLO_DATA_DIR / "labels/train" / f'{Path(row["file_name"]).stem}.txt'
    for line in path.read_text().splitlines():
        points = np.array(line.split()[1:], np.float32).reshape(-1, 2) * [
            IMAGE_SIZE,
            IMAGE_SIZE,
        ]
        cv2.fillPoly(target, [np.rint(points).astype(np.int32)], 1)
    overlay(axes[1, 2], preprocess_grayscale(image), [target.astype(bool)])
    intersection = np.logical_and(target, union).sum()
    denominator = np.logical_or(target, union).sum()
    axes[1, 2].set_title(
        f"Exported YOLO target; union IoU={intersection/max(denominator,1):.4f}"
    )
    fig.suptitle(row["file_name"])
    finish_figure(fig, "preprocessing_targets", row["file_name"])

## 5. Train and retain checkpoint candidates

Only calibration images are used for YOLO validation/early stopping. After training, both `best.pt` (YOLO mask mAP selection) and `last.pt` compete on **calibration PQ** with the same postprocessing grid. This avoids assuming mask mAP selects the best competition checkpoint. The report images do not select checkpoints or thresholds. DDP uses both T4s through the standard Ultralytics trainer.


In [ ]:
training_start = time.perf_counter()
if RUN_TRAINING:
    model = YOLO(YOLO_MODEL_NAME)
    train_result = model.train(
        data=str(DATA_YAML),
        cache=YOLO_CACHE_MODE,
        project=str(RUNS_DIR),
        name=RUN_NAME,
        exist_ok=False,
        **TRAIN_OPTIONS
    )
    run_dir = Path(model.trainer.save_dir)
    checkpoint_candidates = [
        run_dir / "weights" / name for name in ["best.pt", "last.pt"]
    ]
    assert all(p.is_file() for p in checkpoint_candidates)
    save_json(run_dir / "run_config.json", RUN_CONFIG)
    del train_result, model
else:
    if not EXISTING_CHECKPOINT:
        raise ValueError("Set EXISTING_CHECKPOINT when RUN_TRAINING=False.")
    checkpoint_candidates = [Path(EXISTING_CHECKPOINT).resolve()]
    run_dir = checkpoint_candidates[0].parent.parent
    previous = json.loads((run_dir / "run_config.json").read_text(encoding="utf-8"))
    if previous != RUN_CONFIG:
        raise ValueError(
            "Checkpoint transform/targets/split/config differs from this run."
        )
training_seconds = time.perf_counter() - training_start
gc.collect()
if GPU_IDS:
    torch.cuda.empty_cache()
print("Checkpoint candidates:", checkpoint_candidates)
history_path = run_dir / "results.csv"
if history_path.is_file():
    history = pd.read_csv(history_path)
    history.columns = history.columns.str.strip()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for column in ["train/seg_loss", "val/seg_loss"]:
        if column in history:
            axes[0].plot(np.arange(1, len(history) + 1), history[column], label=column)
    for column in ["metrics/mAP50(M)", "metrics/mAP50-95(M)"]:
        if column in history:
            axes[1].plot(np.arange(1, len(history) + 1), history[column], label=column)
    for ax in axes:
        ax.set_xlabel("Epoch")
        ax.grid(alpha=0.2)
        if ax.lines:
            ax.legend()
    axes[0].set_title("Mask loss")
    axes[1].set_title("Calibration mask mAP (not PQ)")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 6. One inference and postprocessing implementation

Native binary masks are encoded directly; no polygon roundtrip. Confidence-filtered predictions are **unioned, then split into 8-connected components**, exactly as in the original 0.36 notebook. Each pixel belongs to at most one predicted filament. The former `instances` candidate was invalid for this competition because near-duplicate suppression still allowed partial overlaps; it has been removed. Old in-memory configurations with that name are safely routed through union/components too.

Calibration selects only confidence, minimum component area, disk clipping and checkpoint. Validation, report, submission and previews all use the same disjoint masks. Cached raw predictions can overlap; only final component masks reach the CSV. A separate audit of the written RLEs rejects even one overlapping pixel before publishing the submission.


In [ ]:
def encode_mask(mask):
    return mask_utils.encode(np.asfortranarray(mask, dtype=np.uint8))


def json_rle(rle):
    return {
        "size": list(rle["size"]),
        "counts": (
            rle["counts"].decode("ascii")
            if isinstance(rle["counts"], bytes)
            else rle["counts"]
        ),
    }


def native_rle(rle):
    return {
        "size": list(rle["size"]),
        "counts": (
            rle["counts"].encode("ascii")
            if isinstance(rle["counts"], str)
            else rle["counts"]
        ),
    }


def decode_mask(rle):
    return mask_utils.decode(native_rle(rle)).astype(bool)


def inference_cache_dir(checkpoint):
    signature = dict(
        weights=file_sha256(checkpoint),
        run=RUN_FINGERPRINT,
        imgsz=IMAGE_SIZE,
        conf=min(CONFIDENCE_CANDIDATES),
        iou=PREDICT_NMS_IOU,
        max_det=MAX_DETECTIONS,
        retina_masks=True,
        nms=None,
    )
    root = EXPERIMENT_DIR / "prediction_cache" / fingerprint(signature)[:16]
    root.mkdir(parents=True, exist_ok=True)
    return root


def cached_predict(model, path, cache_dir):
    path = Path(path)
    destination = cache_dir / f"{path.stem}.json"
    stat = path.stat()
    source_signature = [str(path.resolve()), stat.st_size, stat.st_mtime_ns]
    if destination.is_file():
        cached = json.loads(destination.read_text(encoding="utf-8"))
        if cached["source"] == source_signature:
            return cached
    image = read_gray(path)
    processed = preprocess_grayscale(image)
    source = cv2.cvtColor(processed, cv2.COLOR_GRAY2BGR)
    result = model.predict(
        source=source,
        imgsz=IMAGE_SIZE,
        conf=min(CONFIDENCE_CANDIDATES),
        iou=PREDICT_NMS_IOU,
        retina_masks=True,
        max_det=MAX_DETECTIONS,
        device=INFERENCE_DEVICE,
        batch=1,
        nms=None,
        augment=False,
        save=False,
        verbose=False,
    )[0]
    instances = []
    if result.masks is not None:
        if result.boxes is None or len(result.masks.data) != len(result.boxes.conf):
            raise ValueError("Mask/confidence mismatch.")
        # Convert on GPU before transferring; never allocate a full float32 CPU stack.
        for tensor, confidence in zip(result.masks.data, result.boxes.conf):
            mask = tensor.detach().to(dtype=torch.uint8).cpu().numpy()
            if mask.shape != image.shape:
                raise ValueError(
                    f"Native mask decode failed: {mask.shape} != {image.shape}"
                )
            if mask.any():
                instances.append(
                    {
                        "rle": json_rle(encode_mask(mask)),
                        "confidence": float(confidence.item()),
                    }
                )
    cached = dict(
        source=source_signature,
        file_name=path.name,
        disk=estimate_disk(image),
        instances=instances,
    )
    save_json(destination, cached)
    return cached


def component_rles(binary):
    count, labels, stats, _ = cv2.connectedComponentsWithStats(
        binary.astype(np.uint8), connectivity=8
    )
    rles = []
    for i in range(1, count):
        # One full-size uint8 temporary at a time, never a float mask stack.
        rles.append(encode_mask(labels == i))
    return rles




def base_postprocessing(cached, confidence, mode, clip_to_disk):
    # Legacy selected configurations may contain mode="instances". Always use
    # the original union -> connected-components policy, including for them.
    if mode not in {"union_cc", "instances"}:
        raise ValueError(f"Unknown postprocessing mode: {mode}")
    selected = [i for i in cached["instances"] if i["confidence"] >= confidence]
    rles = [native_rle(i["rle"]) for i in selected]
    if not rles:
        return []
    combined = mask_utils.merge(rles)
    if clip_to_disk and cached["disk"] is not None:
        support = encode_mask(disk_support((IMAGE_SIZE, IMAGE_SIZE), cached["disk"]))
        combined = mask_utils.merge([combined, support], intersect=True)
    # Component labels partition this union: no pixel can belong to two outputs.
    return component_rles(mask_utils.decode(combined))


def filter_rles(rles, min_area):
    rles = [r for r in rles if int(mask_utils.area(r)) >= max(1, min_area)]

    def component_key(rle):
        x, y, _, _ = mask_utils.toBbox(rle)
        # Same order as split_into_filaments in the original 0.36 notebook:
        # descending pixel area, then topmost pixel, then leftmost pixel.
        return (-int(mask_utils.area(rle)), int(y), int(x))

    return sorted(rles, key=component_key)


def canonical_postprocessing_config(config):
    fixed = dict(config)
    if fixed.get("mode", "union_cc") not in {"union_cc", "instances"}:
        raise ValueError(f"Unknown postprocessing mode: {fixed.get('mode')}")
    fixed["mode"] = "union_cc"
    return fixed


def final_prediction(cached, config):
    config = canonical_postprocessing_config(config)
    return filter_rles(
        base_postprocessing(cached, config["confidence"], "union_cc", config["clip_to_disk"]),
        config["min_area"],
    )


# Never allow stale in-memory POSTPROCESS_MODES to re-enable overlapping masks.
POSTPROCESS_GRID = [
    dict(confidence=c, min_area=a, mode="union_cc", clip_to_disk=d)
    for c in CONFIDENCE_CANDIDATES
    for d in CLIP_TO_DISK_CANDIDATES
    for a in AREA_CANDIDATES
]


In [ ]:
def target_rles(image_id):
    rles = [coco.annToRLE(a) for a in coco.loadAnns(coco.getAnnIds(imgIds=[image_id]))]
    return [r for r in rles if int(mask_utils.area(r)) > 0]


def score_rles(predictions, targets):
    n_pred, n_gt = len(predictions), len(targets)
    overlaps = (
        mask_utils.iou(predictions, targets, [0] * n_gt).T
        if n_pred and n_gt
        else np.zeros((n_gt, n_pred), np.float64)
    )
    hits = overlaps > PQ_MATCH_IOU
    # Exactly the organizers' published self-evaluation logic, including overlap cases.
    tp = int(hits.sum())
    fp = int((hits.sum(axis=0) == 0).sum())
    fn = int((hits.sum(axis=1) == 0).sum())
    iou_sum = float(overlaps[hits].sum())
    den = tp + 0.5 * (fp + fn)
    pred_union = mask_utils.merge(predictions) if predictions else None
    gt_union = mask_utils.merge(targets) if targets else None
    pred_area = int(mask_utils.area(pred_union)) if pred_union is not None else 0
    gt_area = int(mask_utils.area(gt_union)) if gt_union is not None else 0
    intersection = (
        int(mask_utils.area(mask_utils.merge([pred_union, gt_union], intersect=True)))
        if pred_area and gt_area
        else 0
    )
    return dict(
        tp=tp,
        fp=fp,
        fn=fn,
        matched_iou_sum=iou_sum,
        pq=iou_sum / den if den else 0.0,
        one_to_many=int((hits.sum(axis=1) > 1).sum()),
        many_to_one=int((hits.sum(axis=0) > 1).sum()),
        n_pred=n_pred,
        n_gt=n_gt,
        pixel_dice=(
            2 * intersection / (pred_area + gt_area) if pred_area + gt_area else 1.0
        ),
        pixel_iou=(
            intersection / (pred_area + gt_area - intersection)
            if pred_area + gt_area
            else 1.0
        ),
    )


def pooled_metrics(rows):
    frame = pd.DataFrame(rows)
    tp, fp, fn = (int(frame[c].sum()) for c in ["tp", "fp", "fn"])
    iou_sum = float(frame.matched_iou_sum.sum())
    denominator = tp + 0.5 * (fp + fn)
    return dict(
        PQ=iou_sum / denominator if denominator else 0.0,
        SQ=iou_sum / tp if tp else 0.0,
        RQ=tp / denominator if denominator else 0.0,
        TP=tp,
        FP=fp,
        FN=fn,
        mean_Dice=float(frame.pixel_dice.mean()),
        mean_IoU=float(frame.pixel_iou.mean()),
        annotator_records=len(frame),
        one_to_many=int(frame.one_to_many.sum()),
        many_to_one=int(frame.many_to_one.sum()),
    )


def score_one_source(cached, image_ids, configs):
    targets = {image_id: target_rles(image_id) for image_id in image_ids}
    output = []
    # Reuse expensive components for all minimum-area candidates of a base setting.
    bases = {}
    for config_idx, cfg in enumerate(configs):
        key = (cfg["confidence"], cfg["mode"], cfg["clip_to_disk"])
        if key not in bases:
            bases[key] = base_postprocessing(cached, *key)
        prediction = filter_rles(bases[key], cfg["min_area"])
        for image_id, gt in targets.items():
            output.append(
                dict(
                    config_idx=config_idx,
                    image_id=image_id,
                    file_name=cached["file_name"],
                    **score_rles(prediction, gt)
                )
            )
    return output


def evaluate_partition(model, frame, configs, cache_dir, description):
    rows = []
    physical = physical_rows(frame)
    # Only NUM_WORKERS pending sources; GPU inference stays on the main thread.
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as pool:
        pending = deque()
        for row in tqdm(physical, desc=description):
            cached = cached_predict(
                model, TRAIN_IMAGE_DIR / row["file_name"], cache_dir
            )
            pending.append(
                pool.submit(score_one_source, cached, row["image_ids"], configs)
            )
            if len(pending) >= NUM_WORKERS:
                rows.extend(pending.popleft().result())
        while pending:
            rows.extend(pending.popleft().result())
    per_record = pd.DataFrame(rows)
    summaries = [
        dict(
            config_idx=i,
            **cfg,
            **pooled_metrics(per_record[per_record.config_idx == i].to_dict("records"))
        )
        for i, cfg in enumerate(configs)
    ]
    return per_record, pd.DataFrame(summaries)

## 7. Select on calibration, then freeze and report

The selected score on calibration is optimistic because it is used for selection. **Report PQ** is the separate estimate. Use these report scores sparingly when comparing raw/radial/CLAHE; repeated tuning against the report also turns it into a selection set. Large leaderboard gains are not guaranteed. Per-annotator counts are pooled, rather than averaging per-image PQ.


In [ ]:
validation_start = time.perf_counter()
all_summaries = []
for checkpoint in checkpoint_candidates:
    model = YOLO(str(checkpoint))
    cache_dir = inference_cache_dir(checkpoint)
    per_record, summary = evaluate_partition(
        model,
        calibration_records,
        POSTPROCESS_GRID,
        cache_dir,
        f"Calibration {checkpoint.name}",
    )
    summary["checkpoint"] = str(checkpoint.resolve())
    summary["weights_sha256"] = file_sha256(checkpoint)
    all_summaries.append(summary)
    per_record.to_csv(
        EXPERIMENT_DIR / f"calibration_{checkpoint.stem}_per_record.csv", index=False
    )
    del model
    gc.collect()
    if GPU_IDS:
        torch.cuda.empty_cache()
calibration_summary = pd.concat(all_summaries, ignore_index=True)
calibration_summary = calibration_summary.sort_values(
    "PQ", ascending=False, kind="stable"
).reset_index(drop=True)
calibration_summary.to_csv(EXPERIMENT_DIR / "calibration_grid.csv", index=False)
winner = calibration_summary.iloc[0]
SELECTED_CONFIG = canonical_postprocessing_config(POSTPROCESS_GRID[int(winner.config_idx)])
SELECTED_CHECKPOINT = Path(winner.checkpoint)
SELECTED_CACHE_DIR = inference_cache_dir(SELECTED_CHECKPOINT)
FROZEN_SELECTION = dict(
    checkpoint=str(SELECTED_CHECKPOINT),
    weights_sha256=winner.weights_sha256,
    postprocessing=SELECTED_CONFIG,
    mask_postprocessing_version="union_cc_nonoverlap_v1",
    calibration_PQ=float(winner.PQ),
    run_fingerprint=RUN_FINGERPRINT,
)
save_json(EXPERIMENT_DIR / "selected_pipeline.json", FROZEN_SELECTION)
display(calibration_summary.head(12))
print("Frozen selection:", FROZEN_SELECTION)

model = YOLO(str(SELECTED_CHECKPOINT))
report_per_record, report_summary = evaluate_partition(
    model, report_records, [SELECTED_CONFIG], SELECTED_CACHE_DIR, "Held-out report"
)
report_per_record.to_csv(EXPERIMENT_DIR / "report_per_annotator.csv", index=False)
report_summary.to_csv(EXPERIMENT_DIR / "report_metrics.csv", index=False)
validation_seconds = time.perf_counter() - validation_start
display(report_summary)
save_json(
    EXPERIMENT_DIR / "timings.json",
    dict(training_seconds=training_seconds, validation_seconds=validation_seconds),
)
fig, ax = plt.subplots(figsize=(8, 4))
for (ckpt, mode, clip), frame in calibration_summary.groupby(
    ["checkpoint", "mode", "clip_to_disk"]
):
    best_per_conf = frame.groupby("confidence").PQ.max()
    ax.plot(
        best_per_conf.index,
        best_per_conf.values,
        marker=".",
        label=f"{Path(ckpt).name}, {mode}, clip={clip}",
    )
ax.set(
    xlabel="Confidence",
    ylabel="Calibration PQ",
    title="Best area threshold per curve; selection scores",
)
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()
plt.close(fig)


## 8. Ten validation prediction previews

Evenly spaced **distinct report images**, including any empty predictions. Ground truth shows the first annotator for display only; the metrics above score all annotators separately. The raw returned masks and selected postprocessing are displayed together, plus a detail crop for thin boundaries. All previews use cached model predictions.


In [ ]:
def prediction_preview(path, cached, final_rles, stage, gt_rles=None):
    image = read_gray(path)
    processed = preprocess_grayscale(image)
    raw = [decode_mask(i["rle"]) for i in cached["instances"]]
    final = [decode_mask(r) for r in final_rles]
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.ravel()
    show_gray(axes[0], image)
    axes[0].set_title("Original 2048 image")
    show_gray(axes[1], processed)
    axes[1].set_title(f"Model input: {PREPROCESSING_METHOD}")
    overlay(axes[2], image, raw)
    axes[2].set_title(
        f"Returned masks (conf ≥ {min(CONFIDENCE_CANDIDATES):.2f}): {len(raw)}"
    )
    if gt_rles is not None:
        overlay(axes[3], image, [decode_mask(r) for r in gt_rles])
        axes[3].set_title("GT: one annotator (display only)")
    else:
        merged = np.zeros(image.shape, np.uint8)
        for mask in final:
            merged |= mask.astype(np.uint8)
        axes[3].imshow(merged, cmap="gray", vmin=0, vmax=1)
        axes[3].axis("off")
        axes[3].set_title("Final mask decoded from submission CSV")
    overlay(axes[4], image, final)
    axes[4].set_title(f"Final predicted instances: {len(final)}")
    if final_rles:
        x, y, w, h = mask_utils.toBbox(native_rle(final_rles[0])).astype(int)
        pad = max(40, int(max(w, h) * 0.20))
        x0, x1 = max(0, x - pad), min(IMAGE_SIZE, x + w + pad)
        y0, y1 = max(0, y - pad), min(IMAGE_SIZE, y + h + pad)
        overlay(axes[5], image[y0:y1, x0:x1], [m[y0:y1, x0:x1] for m in final])
        axes[5].set_title("Largest instance: native-pixel detail")
    else:
        show_gray(axes[5], image)
        axes[5].set_title("No predicted filaments")
    fig.suptitle(f"{Path(path).name} | {SELECTED_CONFIG}", fontsize=11)
    finish_figure(fig, stage, Path(path).name)


for row in pick_evenly(physical_rows(report_records), PREDICTION_PREVIEW_COUNT):
    path = TRAIN_IMAGE_DIR / row["file_name"]
    cached = cached_predict(model, path, SELECTED_CACHE_DIR)
    prediction_preview(
        path,
        cached,
        final_prediction(cached, SELECTED_CONFIG),
        "validation_predictions",
        target_rles(row["image_ids"][0]),
    )

## 9. Competition submission — original 0.36 file format

The output contract is copied from `solar-filament-yolo26-full-res-0-36.ipynb`: the same `mask_to_rle` function, `csv.writer(f)` default settings, header `filament_id,segmentation_rle`, and IDs constructed as `<unchanged JPEG stem>_<counter starting at 1>`. Each row contains only a filament ID and compressed COCO RLE counts. No index, size dictionary or extra columns are added. Components use the original numbering order: descending area, then topmost pixel, then leftmost pixel. Empty images contribute no rows.

The default filename and location also match the original: `Path.cwd() / "solar_filament_yolo26s_submission.csv"`. String paths such as `"submission.csv"` are supported too. Preprocessing and calibrated mask postprocessing determine the predicted masks; serialization uses the original encoder and writer unchanged. Counts are decoded to UTF-8 exactly as in the original, without manually adding quotation marks.

The optimized pipeline still writes incrementally, audits every RLE, records every processed test image, and publishes the final CSV only when the entire test set succeeds. Diagnostic JSON files remain separate. Upload only `solar_filament_yolo26s_submission.csv` from the working directory.

All selected masks are unioned and split into disjoint components before RLE export, matching the original notebook. The final CSV audit verifies zero shared pixels between masks from the same image. The CSV columns and encoder remain unchanged.


In [ ]:
def mask_to_rle(binary_mask):
    encoded = mask_utils.encode(np.asfortranarray(binary_mask.astype(np.uint8)))
    return encoded["counts"].decode("utf-8")


def get_test_image_paths():
    paths = [
        path
        for path in TEST_IMAGE_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg"}
    ]
    return sorted(paths, key=lambda path: path.name)


def test_image_paths():
    return get_test_image_paths()


def submission_rows_for_source(path_stem, rles):
    # The masks come from the same selected postprocessing used in validation.
    # Encode each native mask with the exact original encoder and ID loop.
    components = (decode_mask(rle) for rle in rles)
    rows = []
    for filament_counter, component in enumerate(components, start=1):
        if component.shape != (2048, 2048):
            raise ValueError("Submission masks must be exactly 2048 x 2048.")
        rows.append((f"{path_stem}_{filament_counter}", mask_to_rle(component)))
    return rows


def create_submission_parallel(output_path=SUBMISSION_PATH):
    # Accept both strings and Path objects, as open() did in the original.
    output_path = Path(output_path)
    paths = get_test_image_paths()
    if not paths:
        raise FileNotFoundError(f"No .jpg/.jpeg images found in {TEST_IMAGE_DIR}")
    if len({path.stem for path in paths}) != len(paths):
        raise ValueError("Test images must have unique filename stems.")
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    processed = {}
    seen_filament_ids = set()
    start_time = time.perf_counter()
    export_config = canonical_postprocessing_config(SELECTED_CONFIG)
    export_selection = dict(FROZEN_SELECTION)
    export_selection["postprocessing"] = export_config
    export_selection["mask_postprocessing_version"] = "union_cc_nonoverlap_v1"
    if export_config != SELECTED_CONFIG:
        # A previous score for overlapping instances does not describe this output.
        export_selection.pop("calibration_PQ", None)
        export_selection["recalibration_needed"] = True
        print("Legacy instance mode replaced by disjoint union/components. Recalibrate for updated PQ.", flush=True)

    with open(temporary, mode="w", newline="", encoding="utf-8") as f:
        # Exactly the original writer: default dialect, quoting and CRLF ending.
        writer = csv.writer(f)
        writer.writerow(["filament_id", "segmentation_rle"])
        for path in tqdm(paths, desc="Submission"):
            cached = cached_predict(model, path, SELECTED_CACHE_DIR)
            rles = final_prediction(cached, export_config)
            worker_rows = submission_rows_for_source(path.stem, rles)
            for fid, rle in worker_rows:
                if fid in seen_filament_ids:
                    raise ValueError(f"Duplicate filament ID detected: {fid}")
                seen_filament_ids.add(fid)
                writer.writerow([fid, rle])
            processed[path.stem] = len(worker_rows)

    assert set(processed) == {path.stem for path in paths}
    audit = audit_submission(temporary, processed)
    os.replace(temporary, output_path)
    save_json(EXPERIMENT_DIR / "test_inference_log.json", processed)
    audit.update(
        sha256=file_sha256(output_path),
        inference_seconds=time.perf_counter() - start_time,
        csv=str(output_path),
        pipeline=export_selection,
        submission_format="original_0_36",
    )
    save_json(EXPERIMENT_DIR / "submission_audit.json", audit)
    print(
        f"\n[SUCCESS] Submission complete!\nProcessed: {len(paths):,} images"
        f"\nFilament rows written: {audit['submitted_filaments']:,}\nSaved: {output_path}",
        flush=True,
    )
    return audit


def create_submission(output_path=None):
    # Compatibility with the previous call name; serialization stays original.
    return create_submission_parallel(SUBMISSION_PATH if output_path is None else output_path)


def audit_submission(path, processed):
    seen = set()
    counts = {stem: 0 for stem in processed}
    # Compact accumulated RLE per image handles interleaved image rows as well.
    occupied_rles = {}
    with open(path, newline="", encoding="utf-8") as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ["filament_id", "segmentation_rle"]
        for row in reader:
            assert set(row) == {"filament_id", "segmentation_rle"} and None not in row.values()
            filament_id = row["filament_id"]
            stem, tail = filament_id.rsplit("_", 1)
            assert stem in processed and tail.isdigit() and int(tail) > 0 and filament_id not in seen
            seen.add(filament_id)
            assert row["segmentation_rle"], "Empty RLE counts."
            rle = native_rle({"size": [2048, 2048], "counts": row["segmentation_rle"]})
            mask = decode_mask(rle)
            assert mask.shape == (2048, 2048) and mask.any()
            assert mask_to_rle(mask) == row["segmentation_rle"], "Original-encoder RLE roundtrip failed."
            if stem in occupied_rles:
                intersection = mask_utils.merge([occupied_rles[stem], rle], intersect=True)
                overlap_pixels = int(mask_utils.area(intersection))
                if overlap_pixels:
                    raise ValueError(
                        f"Invalid submission: {filament_id} overlaps an earlier mask for {stem} "
                        f"by {overlap_pixels:,} pixels. The final CSV was not published."
                    )
                occupied_rles[stem] = mask_utils.merge([occupied_rles[stem], rle])
            else:
                occupied_rles[stem] = rle
            counts[stem] += 1
    assert counts == processed, "CSV rows differ from the complete test inference log."
    return dict(
        test_images_processed=len(processed),
        zero_prediction_images=sum(v == 0 for v in processed.values()),
        submitted_filaments=len(seen), decoded_every_rle=True,
        exact_columns=True, unique_ids=True,
        non_overlapping_masks=True, overlap_pixels=0,
    )


if CREATE_SUBMISSION:
    submission_audit = create_submission_parallel()
    print(json.dumps(submission_audit, indent=2))
else:
    print("CREATE_SUBMISSION=False: submission and test galleries skipped.")


## 10. Ten test prediction previews — masks actually written to CSV

Evenly spaced test files are selected without labels or confidence-based cherry-picking. The final masks are decoded from the finished CSV, so previews verify the submitted content. No second model inference is required. Each figure shows the original, processed input, returned predictions, submitted mask/overlay and a detail crop. PNGs are saved under `previews/test_predictions/`.


In [ ]:
if CREATE_SUBMISSION:
    preview_paths = pick_evenly(test_image_paths(), PREDICTION_PREVIEW_COUNT)
    preview_rles = {p.stem: [] for p in preview_paths}
    with Path(SUBMISSION_PATH).open(newline="", encoding="utf-8") as stream:
        for row in csv.DictReader(stream):
            stem = row["filament_id"].rsplit("_", 1)[0]
            if stem in preview_rles:
                preview_rles[stem].append(
                    {"size": [2048, 2048], "counts": row["segmentation_rle"]}
                )
    for path in preview_paths:
        cached = cached_predict(model, path, SELECTED_CACHE_DIR)
        prediction_preview(path, cached, preview_rles[path.stem], "test_predictions")
    print(
        f"Displayed and saved {len(preview_paths)} distinct test prediction previews."
    )


## Outputs and fair comparisons

`solar_filament_yolo26s_submission.csv` is saved in the working directory, exactly as in the original 0.36 notebook. All other outputs are inside the printed fingerprinted experiment directory:

- `run_config.json`, `*_split.csv`, `target_audit.csv`: reproducibility and union-target audit.
- `yolo_runs/.../weights/best.pt` and `last.pt`: trained checkpoint candidates.
- `calibration_grid.csv`, `selected_pipeline.json`: checkpoint/postprocessing selection evidence.
- `report_metrics.csv`, `report_per_annotator.csv`: separate report PQ, SQ, RQ, Dice/IoU and error counts.
- Working-directory `solar_filament_yolo26s_submission.csv`: the **single file to upload**, using the original 0.36 format.
- `test_inference_log.json`, `submission_audit.json`: all test images accounted for, all RLEs verified.
- `previews/validation_predictions/`, `previews/test_predictions/`: 10 examples each (or all, if fewer).

Recommended experiments: **raw → radial → mild CLAHE**, changing only `PREPROCESSING_METHOD` and training each from the same pretrained initialization. Keep the annotator union and all other settings fixed. Compare report PQ and morphology before making a leaderboard submission. Keep raw if an enhancement loses. Do not preprocess only at inference, transfer a raw-trained checkpoint to enhanced inputs without retraining, or infer an expected leaderboard gain from these code changes. The 0.36 notebook is preserved as a separate baseline.

Full-resolution GPU training and competition scoring must be run to determine whether this pipeline improves 0.33/0.36. The implementation improves experimental control, evaluation consistency and selection; it does not establish a new score by itself.

For an already generated CSV, the standalone Quick repair cell at the top writes `*_nonoverlap.csv` without retraining. Fresh exports use union/components throughout and report `non_overlapping_masks=true` in their audit.
